In [ ]:
import SimpleITK as sitk
import numpy as np
from PIL import Image
from scipy import ndimage
import napari
import skimage as sk

In [ ]:
map = 'Maps/Example_map.png'
mask = np.array(Image.open('Masks/Tissue1/example_mask.png').convert('L'))

In [ ]:
viewer = napari.view_image(mask)

In [ ]:
def calculate_spacing(moving_path,fixed_path,grey_value):
    fixed = np.array(Image.open(fixed_path).convert('L'))
    moving = np.array(Image.open(moving_path).convert('L'))
    #get largest region from tissue (excludes the cardinal points) to get bounding box of tissue mask
    moving_binary = moving < 255
    moving_labels = sk.measure.label(moving_binary)
    moving_props = sk.measure.regionprops(moving_labels)
    largest_obj = max(moving_props, key=lambda p: p.area)
    minc,minr,maxc,maxr = largest_obj.bbox
    moving_height_px = maxr - minr
    moving_width_px  = maxc - minc
    #moving_bbox = moving_binary[minc:maxc,minr:maxr]
    #get bounding box of map region
    map_region = fixed * (fixed==grey_value)
    map_region_label = sk.measure.label(map_region)
    map_region_props = sk.measure.regionprops(map_region_label)
    minc,minr,maxc,maxr = map_region_props[0].bbox
    map_height_px = maxr - minr
    map_width_px  = maxc - minc
    # map_region_bbox = map_region[minc:maxc,minr:maxr]
    #calculate_spacing assuming pixels have 1:1 ratio
    spacing_x = map_width_px / moving_width_px
    spacing_y = map_height_px / moving_height_px
    moving_spacing = (spacing_x,spacing_y)
    map_spacing = (round((moving_width_px/map_width_px),1),round((moving_height_px/map_height_px),1))
    return map_spacing




In [ ]:
print(calculate_spacing('Masks/Tissue1/example_mask.png','Maps/Example_map.png',150))

In [ ]:
def calculate_spacing_from_cardinal_points(map_points, mask_points, source_spacing=(1.0, 1.0)):
    """
    Derive map spacing from the physical N→E distance in both images.

    The N→E vector represents the same physical distance in both the map
    and the source — so their physical lengths must be equal after spacing
    is applied. Solving for map_spacing:

        |map_vec_px| * map_spacing = |mask_vec_px * source_spacing|

    This is more reliable than bounding box comparison because:
    - It does not assume the source fills the region bounding box
    - It uses the same landmarks used for rotation/flip
    - It produces isotropic spacing, avoiding angle distortion
    - A large E-point error in compute_orientation_transform directly
      indicates this calculation is wrong

    Parameters
    ----------
    map_points    : dict with 'north' and 'east' as pixel-coordinate arrays
    mask_points   : dict with 'north' and 'east' as pixel-coordinate arrays
    source_spacing: known physical spacing of the source image (x, y)

    Returns
    -------
    (spacing_x, spacing_y) isotropic tuple for the map image
    """
    map_vec_px  = map_points["east"]  - map_points["north"]   # map pixels
    mask_vec_px = mask_points["east"] - mask_points["north"]  # source pixels

    # Physical N→E length in source space
    mask_phys_length = np.linalg.norm(mask_vec_px * np.array(source_spacing))

    # Pixel N→E length in map space
    map_px_length = np.linalg.norm(map_vec_px)

    if map_px_length < 1e-6:
        raise ValueError("Map N→E vector is near zero — cardinal points may be "
                         "identical or detection failed")

    # Isotropic: single spacing value for both axes
    # Non-isotropic would require knowing axis alignment, which we don't
    map_spacing = mask_phys_length / map_px_length
    print(f"  Mask N→E physical length: {mask_phys_length:.4f} source units")
    print(f"  Map  N→E pixel length:    {map_px_length:.4f} px")
    print(f"  Derived map spacing:      {map_spacing:.6f} source-units/px")

    # Sanity: compare to bounding box estimate if wildly different
    return (round(float(map_spacing),1), round(float(map_spacing),1))

In [ ]:
def detect_cardinal_point(image_path, grey_value):
    """
    Detect the centroid of a cardinal point marker by its grey value.
    
    Returns (x, y) in pixel coordinates, or None if not found.
    """
    arr = np.array(Image.open(image_path).convert("L"))
    
    # Create binary mask for this grey value
    mask = (arr * (arr == grey_value)).astype(int)
    
    
    if not mask.any():
        print(f"  WARNING: No pixels found for grey value {grey_value} in {image_path}")
        return None
    
    # Label connected components and get centroid
    labeled_img = sk.measure.label(mask)
    props = sk.measure.regionprops(labeled_img)
    cy,cx = props[0].centroid
    return np.array([cx, cy])  # return as (x, y)

In [ ]:
def get_cardinal_points(image_path, grey_values_dict):
    """
    Extract both cardinal points from an image.
    Returns dict with 'north' and 'east' as (x, y) arrays.
    """
    points = {}
    for direction in ["north", "east"]:
        grey = grey_values_dict[direction]
        pt = detect_cardinal_point(image_path, grey)
        if pt is None:
            raise ValueError(f"Could not find '{direction}' point in {image_path}")
        points[direction] = pt
        print(f"  {direction}: pixel ({pt[0]:.1f}, {pt[1]:.1f})")
    return points

In [ ]:
def compute_flip_transform(map_points, mask_points,
                            fixed_sitk, moving_sitk):
    """
    Determine and apply only the flip needed to orient the moving image
    to match the map, using the N and E cardinal point locations.
    No rotation is applied.

    Detection logic (image coordinates, y increases downward):
        North marker should be visually "above" East  → north_y < east_y
        East  marker should be visually "right" of North → east_x > north_x

    Flip cases:
        north_is_up  and east_is_right  → none
        north_is_up  and not east_is_right → horizontal  (left/right mirror)
        not north_is_up and east_is_right  → vertical    (top/bottom mirror)
        not north_is_up and not east_is_right → both

    Flip matrices are self-inverse (F @ F = I), so the forward and inverse
    transforms use the same matrix — only the translation differs.

    Parameters
    ----------
    map_points   : dict with 'north' and 'east' as (x, y) pixel arrays — fixed image
    mask_points  : dict with 'north' and 'east' as (x, y) pixel arrays — moving image
    fixed_sitk   : SimpleITK image (the map region)
    moving_sitk  : SimpleITK image (the source mask)

    Returns
    -------
    sitk_transform : SimpleITK AffineTransform (inverse convention, zero center)
    chosen_flip    : str — one of 'none', 'horizontal', 'vertical', 'both'
    """
    FLIP_MATRICES = {
        "none":       np.array([[ 1,  0], [ 0,  1]], dtype=float),
        "horizontal": np.array([[-1,  0], [ 0,  1]], dtype=float),
        "vertical":   np.array([[ 1,  0], [ 0, -1]], dtype=float),
        "both":       np.array([[-1,  0], [ 0, -1]], dtype=float),
    }

    # --- Convert pixel coords to physical coords ---
    def to_physical(pt, sitk_img):
        spacing = np.array(sitk_img.GetSpacing())
        origin  = np.array(sitk_img.GetOrigin())
        return pt * spacing + origin

    map_N  = to_physical(map_points["north"],  fixed_sitk)
    map_E  = to_physical(map_points["east"],   fixed_sitk)
    mask_N = to_physical(mask_points["north"], moving_sitk)
    mask_E = to_physical(mask_points["east"],  moving_sitk)

    # --- Detect required flip from cardinal point geometry ---
    # Compare pixel coords directly for orientation — spacing does not
    # affect which direction is "up" or "right"
    north_is_up   = mask_points["north"][1] < mask_points["east"][1]  # smaller y = higher
    east_is_right = mask_points["east"][0]  > mask_points["north"][0] # larger x = righter

    if north_is_up and east_is_right:
        chosen_flip = "none"
    elif north_is_up and not east_is_right:
        chosen_flip = "horizontal"
    elif not north_is_up and east_is_right:
        chosen_flip = "vertical"
    else:
        chosen_flip = "both"

    print(f"  North is up:   {north_is_up}  "
          f"(mask north_y={mask_points['north'][1]:.1f}, east_y={mask_points['east'][1]:.1f})")
    print(f"  East is right: {east_is_right}  "
          f"(mask east_x={mask_points['east'][0]:.1f}, north_x={mask_points['north'][0]:.1f})")
    print(f"  Detected flip: '{chosen_flip}'")

    F = FLIP_MATRICES[chosen_flip]

    # --- Compute translation to align mask N → map N after flip ---
    # Forward: F @ mask_N + t_fwd = map_N
    t_fwd = map_N - F @ mask_N

    # Since F is self-inverse (F @ F = I), A_inv = F
    # Inverse translation: t_inv = -F @ t_fwd
    t_inv = -F @ t_fwd

    # --- Verification ---
    fwd_ok = np.allclose(F @ mask_N + t_fwd, map_N)
    inv_ok = np.allclose(F @ map_N  + t_inv, mask_N)
    print(f"  Forward check (F @ mask_N + t = map_N): {fwd_ok}  "
          f"{np.round(F @ mask_N + t_fwd, 2)} vs {np.round(map_N, 2)}")
    print(f"  Inverse check (F @ map_N + t_inv = mask_N): {inv_ok}  "
          f"{np.round(F @ map_N + t_inv, 2)} vs {np.round(mask_N, 2)}")

    # E-point error — checks that spacing is consistent
    E_check  = F @ mask_E + t_fwd
    e_err    = np.linalg.norm(E_check - map_E)
    print(f"  E-point forward error: {e_err:.4f} physical units "
          f"(should be ~0 if spacing is correct)")
    if e_err > 10:
        print(f"  ⚠ WARNING: Large E-point error — check calculate_spacing_from_cardinal_points")

    # Full 3x3 for reference
    M_inv = np.eye(3)
    M_inv[:2, :2] = F          # self-inverse
    M_inv[:2,  2] = t_inv
    print(f"  Inverse flip matrix (SimpleITK convention):\n{np.round(M_inv, 4)}")

    # --- Build SimpleITK transform ---
    # Zero center: transform applies exactly as  output = F @ input + t_inv
    sitk_transform = sitk.AffineTransform(2)
    sitk_transform.SetMatrix(F.flatten().tolist())  # F = F_inv for flips
    sitk_transform.SetTranslation(t_inv.tolist())
    sitk_transform.SetFixedParameters([0.0, 0.0])

    return sitk_transform, chosen_flip

In [ ]:
def compute_orientation_transform(map_points, mask_points, fixed_sitk, moving_sitk_with_cardinal, flip):
    """
    Compute an AffineTransform aligning mask orientation to the map
    using N and E cardinal points.

    flip options:
        "auto"       — try all four, pick the one requiring least rotation
        "none"       — no flip applied
        "horizontal" — mirror left/right  (negate x)
        "vertical"   — mirror top/bottom  (negate y)
        "both"       — horizontal + vertical (equivalent to 180° rotation)
    """
    # --- Flip matrices in physical space ---
    FLIP_MATRICES = {
        "none":       np.array([[ 1,  0], [ 0,  1]], dtype=float),
        "horizontal": np.array([[-1,  0], [ 0,  1]], dtype=float),
        "vertical":   np.array([[ 1,  0], [ 0, -1]], dtype=float),
        "both":       np.array([[-1,  0], [ 0, -1]], dtype=float),
    }

    # --- Convert pixel coords to physical coords ---
    def to_physical(pt, sitk_img):
        spacing = np.array(sitk_img.GetSpacing())
        origin  = np.array(sitk_img.GetOrigin())
        return pt * spacing + origin

    map_N  = to_physical(map_points["north"],  fixed_sitk)
    map_E  = to_physical(map_points["east"],   fixed_sitk)
    mask_N = to_physical(mask_points["north"], moving_sitk_with_cardinal)
    mask_E = to_physical(mask_points["east"],  moving_sitk_with_cardinal)

    map_vec  = map_E  - map_N
    mask_vec = mask_E - mask_N

    map_angle = np.arctan2(map_vec[1], map_vec[0])
    print(f"  Map  N→E angle: {np.degrees(map_angle):.2f}°")
    print(f"  Mask N→E angle: {np.degrees(np.arctan2(mask_vec[1], mask_vec[0])):.2f}°")

    # --- Compute rotation angle needed after a given flip ---
    def rotation_after_flip(flip_key):
        F = FLIP_MATRICES[flip_key]
        flipped_vec = F @ mask_vec
        flipped_angle = np.arctan2(flipped_vec[1], flipped_vec[0])
        # Wrap to [-180, 180]
        angle = map_angle - flipped_angle
        angle = (angle + np.pi) % (2 * np.pi) - np.pi
        return angle

    # --- Resolve flip mode ---
    if flip == "auto":
        # Try all four options, pick the one requiring the smallest rotation
        candidates = {}
        for key in FLIP_MATRICES:
            angle = rotation_after_flip(key)
            candidates[key] = abs(angle)
            print(f"  [{key:>10}] would need rotation: {np.degrees(angle):+.2f}°")
        
        chosen_flip = min(candidates, key=candidates.get)
        print(f"  Auto-selected flip: '{chosen_flip}' "
              f"(least rotation: {np.degrees(candidates[chosen_flip] if candidates[chosen_flip] < np.pi else -candidates[chosen_flip]):.2f}°)")
    
    elif flip in FLIP_MATRICES:
        chosen_flip = flip
        print(f"  Using specified flip: '{chosen_flip}'")
    
    else:
        raise ValueError(f"flip must be one of {list(FLIP_MATRICES.keys()) + ['auto']}, got '{flip}'")

    # --- Build final transform ---
    F = FLIP_MATRICES[chosen_flip]
    rotation_angle = rotation_after_flip(chosen_flip)

    print(f"  Flip applied:   {chosen_flip}")
    print(f"  Rotation angle: {np.degrees(rotation_angle):.2f}°")

    cos_r = np.cos(rotation_angle)
    sin_r = np.sin(rotation_angle)
    R = np.array([[cos_r, -sin_r],
                  [sin_r,  cos_r]])

    # Combined: rotate after flip
    A = R @ F

    # Forward translation: maps mask_N → map_N
    # A @ mask_N + t_fwd = map_N
    t_fwd = map_N - A @ mask_N

    # ---------------------------------------------------------------
    # CRITICAL: SimpleITK needs the INVERSE transform (fixed → moving)
    # Inverse of  x_fixed = A @ x_moving + t_fwd
    # is          x_moving = A_inv @ x_fixed + t_inv
    # where       t_inv = -A_inv @ t_fwd
    # ---------------------------------------------------------------
    A_inv = np.linalg.inv(A)
    t_inv = -A_inv @ t_fwd

    # Verification
    check_fwd = A     @ mask_N + t_fwd
    check_inv = A_inv @ map_N  + t_inv
    print(f"  Forward check  — A @ mask_N + t should == map_N:  "
          f"{np.allclose(check_fwd, map_N)}  "
          f"({np.round(check_fwd,1)} vs {np.round(map_N,1)})")
    print(f"  Inverse check  — A_inv @ map_N + t_inv should == mask_N: "
          f"{np.allclose(check_inv, mask_N)}  "
          f"({np.round(check_inv,1)} vs {np.round(mask_N,1)})")
    #Verify the E point transforms correctly - catches scale errors
    map_E_check  = A     @ mask_E + t_fwd
    mask_E_check = A_inv @ map_E  + t_inv
    e_fwd_err  = np.linalg.norm(map_E_check  - map_E)
    e_inv_err  = np.linalg.norm(mask_E_check - mask_E)
    print(f"  E-point forward error: {e_fwd_err:.3f} physical units "
          f"(should be ~0 if spacing is correct)")
    print(f"  E-point inverse error: {e_inv_err:.3f} physical units")
    if e_fwd_err > 10:
        print(f"  ⚠ WARNING: Large E-point error suggests spacing mismatch. "
              f"Check calculate_spacing output.")
        
    # Full 3x3 for reference (inverse convention)
    M_inv_full = np.eye(3)
    M_inv_full[:2, :2] = A_inv
    M_inv_full[:2, 2]  = t_inv
    print(f"  Inverse orientation matrix (SimpleITK convention):\n"
          f"{np.round(M_inv_full, 4)}")

    # Center on moving image centroid (SimpleITK fixed parameters)
    # moving_center = (np.array(moving_sitk_with_cardinal.GetSize())  / 2.0) * \
    #                  np.array(moving_sitk_with_cardinal.GetSpacing())

    sitk_transform = sitk.AffineTransform(2)
    sitk_transform.SetMatrix(A_inv.flatten().tolist())
    sitk_transform.SetTranslation(t_inv.tolist())
    sitk_transform.SetFixedParameters([0,0])

    return sitk_transform, chosen_flip, rotation_angle

In [ ]:
def register_region_to_map(orientation,fixed_sitk, moving_sitk_without_cardinal, moving_sitk_with_cardinal, map_points, mask_points, label_id,flip='auto'):
    """
    Full pipeline:
    1. Coarse orientation from cardinal points
    2. Fine affine registration
    3. Returns composite transform and 3x3 matrix
    """
    print(f"\n--- Label {label_id} ---")
    
    # Step 1: Coarse orientation
    if orientation == 'cardinal':
        print("Computing orientation from cardinal points...")
        transform, was_flipped, rotation_deg = compute_orientation_transform(
            map_points, mask_points, fixed_sitk, moving_sitk_with_cardinal,flip
        )
        print(f'Flip: {was_flipped}; Rotation: {rotation_deg}')
    if orientation == 'flip':
        print("Computing orientation with only flips...")
        transform, chosen_flip = compute_flip_transform(map_points,mask_points,fixed_sitk,moving_sitk_without_cardinal)
        print(f'Flip: {chosen_flip}')
    #Step 2: Convert to distance maps for registration
    def to_distance_map(img):
        dist = sitk.SignedMaurerDistanceMap(
            sitk.Cast(img, sitk.sitkUInt8),
            insideIsPositive=True,
            squaredDistance=False,
            useImageSpacing=True
        )
        return dist
    
    fixed_dist  = to_distance_map(fixed_sitk)
    moving_dist = to_distance_map(moving_sitk_without_cardinal)
    
    # Step 3: Fine registration starting from orientation transform

    centering_transform = sitk.CenteredTransformInitializer(
        fixed, moving,
        sitk.AffineTransform(2),
        sitk.CenteredTransformInitializerFilter.MOMENTS
    )
    
    combined_transform = sitk.CompositeTransform([centering_transform,transform])

    registration = sitk.ImageRegistrationMethod()
    registration.SetMetricAsMeanSquares()
    registration.SetInitialTransform(combined_transform, inPlace=False)
    
    registration.SetOptimizerAsLBFGS2(
        solutionAccuracy=1e-5,
        numberOfIterations=500,
        deltaConvergenceTolerance=0.01
    )
    registration.SetOptimizerScalesFromPhysicalShift()
    registration.SetShrinkFactorsPerLevel(shrinkFactors=[8, 4, 2, 1])
    registration.SetSmoothingSigmasPerLevel(smoothingSigmas=[4, 2, 1, 0])
    registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    registration.SetInterpolator(sitk.sitkLinear)
    
    print("Running fine registration...")
    final_transform = registration.Execute(fixed_sitk, moving_sitk_without_cardinal)
    print(f"  Metric: {registration.GetMetricValue():.6f}")
    print(f"  Stop:   {registration.GetOptimizerStopConditionDescription()}")
    
    return final_transform


def extract_matrix(transform):
    """Extract 3x3 homogeneous matrix from SimpleITK transform."""
    if transform.GetName() == "CompositeTransform":
        n = transform.GetNumberOfTransforms()
        affine = transform.GetNthTransform(n - 1)
    else:
        affine = transform
    
    matrix_2x2 = np.array(affine.GetMatrix()).reshape(2, 2)
    translation = np.array(affine.GetTranslation())
    center      = np.array(affine.GetFixedParameters()[:2])
    
    translation_full = -matrix_2x2 @ center + center + translation
    
    M = np.eye(3)
    M[:2, :2] = matrix_2x2
    M[:2, 2]  = translation_full
    return M

In [ ]:
def load_binary_with_spacing_and_no_cardinal(path, spacing,north,east):
    """Load a mask image as binary SimpleITK image with physical spacing set."""
    img = Image.open(path).convert("L")
    arr = np.array(img)
    arr[arr==north] = 255
    arr[arr==east] = 255
    binary = (arr < 255).astype(np.float32)
    
    sitk_img = sitk.GetImageFromArray(binary)
    sitk_img.SetSpacing(spacing)  # critical — sets physical scale
    
    return sitk_img

def load_binary_with_spacing_and_with_cardinal(path, spacing):
    """Load a mask image as binary SimpleITK image with physical spacing set."""
    img = Image.open(path).convert("L")
    arr = np.array(img)
    binary = (arr < 255).astype(np.float32)
    
    sitk_img = sitk.GetImageFromArray(binary)
    sitk_img.SetSpacing(spacing)  # critical — sets physical scale
    
    return sitk_img

def extract_region_with_spacing(label_mask_path, label_value, spacing, output_path=None):
    """Extract a single region from labeled map, with physical spacing set."""
    img = Image.open(label_mask_path).convert("L")
    arr = np.array(img)
    
    binary = (arr == label_value).astype(np.float32)

    sitk_img = sitk.GetImageFromArray(binary)
    sitk_img.SetSpacing(spacing)  # master map spacing
    
    if output_path:
        sitk.WriteImage(sitk_img, output_path)
    
    return sitk_img

In [ ]:
img = np.array(Image.open("Masks/Tissue1/example_mask.png").convert('L'))
img[img==200] = 255
img[img==110] = 255
binary = (img < 255).astype(np.float32)

viewer = napari.view_image(binary)

In [ ]:
print(GREY_VALUES[3])

In [ ]:
source_images = {
    # 1: {"path": "Masks/205892_R1.png"},
    # 2: {"path": "Masks/205892_R2.png"},
    3: {"path": "Masks/Tissue1/example_mask.png"},
}

MAP_PATH = 'Maps/Example_map.png'
SOURCE_SPACING = (1.0,1.0)
GREY_VALUES = {
    # Cardinal points
    "north": 200,   # replace with your actual grey values after
    "east":  110,   # greyscale conversion of your chosen colors
    # Region labels (map only)
    1: 29,   # e.g., blue   (0,0,255)   → L = 0.114*255 ≈ 29... 
    2: 76,  # e.g., green  (0,255,0)   → L value
    3: 150,   # e.g., red    (255,0,0)   → L value
    "exclude":  255,  # white hole
}

# Detect cardinal points in the map once
print("Detecting cardinal points in map...")
map_points = get_cardinal_points(MAP_PATH, GREY_VALUES)

# --- Run per-region ---
transforms = {}
matrices   = {}
metadata   = {}
orientation = 'flip'
for label_id, source_info in source_images.items():
    # moving_spacing,map_spacing = calculate_spacing(source_info["path"],MAP_PATH,GREY_VALUES,label_id)
    # MAP_SPACING = map_spacing   # update when known
    # SOURCE_SPACING = (1.0,1.0)  # your known source spacing
   
    # Detect cardinal points in this source mask and calculate relative map spacing
    print(f"Detecting cardinal points in source {label_id}...")
    mask_points = get_cardinal_points(source_info["path"], GREY_VALUES)
    print(f"Calculating map spacing...")
    map_points = get_cardinal_points(MAP_PATH,GREY_VALUES)
    map_spacing = calculate_spacing(source_info['path'],MAP_PATH,GREY_VALUES[label_id])
    print(f"  Map spacing: {map_spacing}")

    # Load images with spacing
    fixed  = extract_region_with_spacing(MAP_PATH, GREY_VALUES[label_id], map_spacing)
    moving_cardinal = load_binary_with_spacing_and_with_cardinal(source_info["path"], SOURCE_SPACING)
    moving_nocardinal = load_binary_with_spacing_and_no_cardinal(source_info["path"], SOURCE_SPACING,GREY_VALUES['north'],GREY_VALUES['east'])
    #check physical extents
    fixed_extent  = np.array(fixed.GetSize())  * np.array(fixed.GetSpacing())
    moving_extent = np.array(moving.GetSize()) * np.array(moving.GetSpacing())
    print(f"\nLabel {label_id}")
    print(f"  Fixed  physical extent: {fixed_extent}  px size: {fixed.GetSize()}")
    print(f"  Moving physical extent: {moving_extent}  px size: {moving.GetSize()}")
    # Register
    final_transform = register_region_to_map(orientation, fixed, moving_nocardinal, moving_cardinal, map_points, mask_points, label_id)
    
    # Extract and save matrix
    # M = extract_matrix(final_transform)
    # transforms[label_id] = final_transform
    # matrices[label_id]   = M
    # metadata[label_id]   = {
    #     "was_flipped": bool(was_flipped),
    #     "initial_rotation_deg": float(np.degrees(rotation)),
    #     "final_metric": float(final_transform.GetMetricValue())
    # }
    
    # np.savetxt(f"affine_matrix_label_{label_id}.csv", M,
    #            delimiter=",", header="col1,col2,col3", comments="")
    # sitk.WriteTransform(final_transform, f"transform_label_{label_id}.tfm")
    # print(f"  Saved matrix and transform for label {label_id}")

# Save metadata
# import json
# with open("registration_metadata.json", "w") as f:
#     json.dump(metadata, f, indent=2)
# print("\nAll registrations complete.")

In [ ]:
def load_as_binary_sitk(path):
    """Load any mask/annotation image as a binary SimpleITK image."""
    img = Image.open(path).convert("L")
    arr = np.array(img)
    binary = (arr < 255).astype(np.float32)
    return sitk.GetImageFromArray(binary)

In [ ]:
import matplotlib.pyplot as plt

map_img = Image.open('Maps/Example_map.png').convert('L')
map_arr = np.array(map_img)
# fixed_array = sitk.GetArrayFromImage(map)
fig, axes = plt.subplots()
#axes.imshow(map_arr, cmap="gray")

for i, label_id in enumerate(source_images.keys()):
    fixed = extract_region_with_spacing(MAP_PATH, GREY_VALUES[label_id], MAP_SPACING)
    #moving_spacing = calculate_spacing(source_info["path"],MAP_PATH,GREY_VALUES,label_id)
    moving = load_as_binary_sitk(source_images[label_id]['path'])
    transform = sitk.ReadTransform(f'transform_label_{label_id}.tfm')
    resampled = sitk.Resample(
        moving, fixed, transform,
        sitk.sitkNearestNeighbor, 0.0, moving.GetPixelID()
    )
    
    fixed_arr   = sitk.GetArrayFromImage(fixed)
    moved_arr   = sitk.GetArrayFromImage(resampled)
    axes.imshow(fixed_arr, cmap="gray")
    axes.imshow(moved_arr,cmap="Reds",alpha=0.5)

plt.tight_layout()
plt.savefig("registration_verification_fullmap.png", dpi=150)
plt.show()